# Delta Demo — Episode 16: Shallow Clone
### "Copy a Delta Table Without Copying Data — Does It Survive VACUUM? 😲"

---
**Prerequisites:** None

**Runtime:** Databricks Free Edition

**Run Mode:** Run All

**Safe to rerun:** Yes

**Creates its own demo tables:** `employees_ep16_source`, `employees_ep16_clone`

**Deletes only its own demo data:** Yes

---

**A deliberate architecture change from every episode since 10:** `SHALLOW CLONE` only works on Unity Catalog **managed** tables in this environment — path-based tables in a Volume don't support it (confirmed earlier in this series with a real `CANNOT_SHALLOW_CLONE_NON_UC_MANAGED_TABLE_AS_SOURCE_OR_TARGET` error). So this episode uses real managed tables, and — as a direct consequence — **the result is genuinely different from the commonly-cited Shallow Clone warning**, confirmed against real Databricks documentation.

**Learning Outcome:** By the end of this episode, viewers should understand what Shallow Clone actually copies (metadata + log, zero data), and know precisely why Unity Catalog managed-table shallow clones behave differently under VACUUM than the classic/Hive-metastore pattern most tutorials describe.

**Core Question:** Can you create a full, independently-queryable copy of a Delta table without physically duplicating its data — and does it survive if someone cleans up the original afterward?

### Today's Journey
✔ Create a baseline managed table

↓

✔ Shallow clone it — instantly

↓

✔ Prove, from real metadata, that zero bytes were copied

↓

✔ Modify the clone — confirm the source is untouched

↓

✔ Force a REAL physical deletion on the source (confirmed non-zero)

↓

✔ Discover the clone still works — and the documented reason why

# =====================================================
# STEP 0 — Setup (Self-Contained Reset)
# =====================================================
These are real catalog-managed tables — cleanup is `DROP TABLE`, not `rm -rf`.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.delta_demo;

In [0]:
%sql
DROP TABLE IF EXISTS workspace.delta_demo.employees_ep16_clone;
DROP TABLE IF EXISTS workspace.delta_demo.employees_ep16_clone_fresh;
DROP TABLE IF EXISTS workspace.delta_demo.employees_ep16_source;

# =====================================================
# STEP 1 — Create the Baseline (Managed Table, 5 Records)
# =====================================================

In [0]:
%sql
CREATE TABLE workspace.delta_demo.employees_ep16_source (
  eno INT, ename STRING, sal DECIMAL(10,2)
) USING DELTA;

In [0]:
%sql
INSERT INTO workspace.delta_demo.employees_ep16_source VALUES
(1, 'Ravi', 25000),
(2, 'Sridevi', 23000),
(3, 'Uma', 35000),
(4, 'Srik', 32000),
(5, 'Kanth', 28000);

num_affected_rows,num_inserted_rows
5,5


### Verify Baseline

In [0]:
%sql
SELECT * FROM workspace.delta_demo.employees_ep16_source ORDER BY eno;

eno,ename,sal
1,Ravi,25000.00
2,Sridevi,23000.00
3,Uma,35000.00
4,Srik,32000.00
5,Kanth,28000.00


In [0]:
%sql
DESCRIBE DETAIL workspace.delta_demo.employees_ep16_source;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,18c4b15c-1e0b-4c1b-909b-6df814121574,workspace.delta_demo.employees_ep16_source,null,,2026-08-02T12:58:48.086Z,2026-08-02T12:58:59.000Z,List(),List(),1,1307,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, delta.enableRowTracking -> true, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-2a8a2f28-7ca6-4390-b459-b62622a45865, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-d23be071-c8a8-4dd2-82ac-adcf1224c651)",3,7,"List(appendOnly, deletionVectors, domainMetadata, invariants, rowTracking)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
DESCRIBE HISTORY workspace.delta_demo.employees_ep16_source;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2026-08-02T12:58:59.000Z,72738353004153,srikanth.enterprise.ai@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(1277312088332833),1666527b-4098-45ec-9c41-fb7f8500484a,0802-120848-cqy4o1ld-v2n,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 5, numOutputBytes -> 1307)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
0,2026-08-02T12:58:49.000Z,72738353004153,srikanth.enterprise.ai@gmail.com,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-2a8a2f28-7ca6-4390-b459-b62622a45865"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-d23be071-c8a8-4dd2-82ac-adcf1224c651""}, statsOnLoad -> false)",null,List(1277312088332833),b8d94113-eb18-488a-97af-6629826dcf74,0802-120848-cqy4o1ld-v2n,null,WriteSerializable,true,Map(),null,Databricks-Runtime/18.x-aarch64-photon-scala2.13


# =====================================================
# STEP 2 — Shallow Clone
# =====================================================

In [0]:
%sql
--"Same rows does NOT mean same Parquet files were copied."
CREATE TABLE workspace.delta_demo.employees_ep16_clone
SHALLOW CLONE workspace.delta_demo.employees_ep16_source;

### Verify — the Clone Has Identical Data, Immediately

In [0]:
%sql
SELECT * FROM workspace.delta_demo.employees_ep16_clone ORDER BY eno;

eno,ename,sal
1,Ravi,25000.00
2,Sridevi,23000.00
3,Uma,35000.00
4,Srik,32000.00
5,Kanth,28000.00


In [0]:
%python
source_rows = spark.table("workspace.delta_demo.employees_ep16_source").orderBy("eno").collect()
clone_rows = spark.table("workspace.delta_demo.employees_ep16_clone").orderBy("eno").collect()

if source_rows == clone_rows:
    print("✅ VERIFIED: clone's data is identical to the source, immediately.")
else:
    print("❌ NOT VERIFIED — investigate.")

✅ VERIFIED: clone's data is identical to the source, immediately.


# =====================================================
# STEP 3 — The Evidence: Was Any Data Actually Copied?
# =====================================================

In [0]:
%sql
DESCRIBE HISTORY workspace.delta_demo.employees_ep16_clone;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-08-02T13:03:23.000Z,72738353004153,srikanth.enterprise.ai@gmail.com,CLONE,"Map(source -> workspace.delta_demo.employees_ep16_source, sourceVersion -> 1, isShallow -> true)",null,List(1277312088332833),99ca3855-2a15-4684-9f9b-64595cad3b6a,0802-120848-cqy4o1ld-v2n,-1,Serializable,false,"Map(removedFilesSize -> 0, numRemovedFiles -> 0, sourceTableSize -> 1307, numCopiedFiles -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, copiedFilesSize -> 0, sourceNumOfFiles -> 1)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13


In [0]:
%python
clone_history = spark.sql("DESCRIBE HISTORY workspace.delta_demo.employees_ep16_clone")
clone_create = clone_history.orderBy("version").first()
metrics = dict(clone_create['operationMetrics'])

print(f"Operation: {clone_create['operation']}")
print("Full operationMetrics:")
for k, v in metrics.items():
    print(f"  {k}: {v}")

Operation: CLONE
Full operationMetrics:
  removedFilesSize: 0
  numRemovedFiles: 0
  sourceTableSize: 1307
  numCopiedFiles: 0
  numDeletionVectorsAdded: 0
  numDeletionVectorsRemoved: 0
  copiedFilesSize: 0
  sourceNumOfFiles: 1


### VERIFY — Zero Files, Zero Bytes, Physically Copied

In [0]:
%python
#"Now let's see what happens when only the clone changes."
num_copied_files = int(metrics.get('numCopiedFiles', -1))
copied_files_size = int(metrics.get('copiedFilesSize', -1))
source_num_files = int(metrics.get('sourceNumOfFiles', -1))

print(f"sourceNumOfFiles: {source_num_files}")
print(f"numCopiedFiles: {num_copied_files}")
print(f"copiedFilesSize: {copied_files_size}")

if num_copied_files == 0 and copied_files_size == 0:
    print("\n✅ VERIFIED: SHALLOW CLONE copied 0 files and 0 bytes,")
    print("   even though the source has real data files. The clone's")
    print("   log points at the SOURCE's physical files — nothing was")
    print("   duplicated. This is the entire trick.")
else:
    print("\n❌ NOT VERIFIED — check operationMetrics above.")

sourceNumOfFiles: 1
numCopiedFiles: 0
copiedFilesSize: 0

✅ VERIFIED: SHALLOW CLONE copied 0 files and 0 bytes,
   even though the source has real data files. The clone's
   log points at the SOURCE's physical files — nothing was
   duplicated. This is the entire trick.


# =====================================================
# STEP 4 — Modify the Clone: Does the Source Notice?
# =====================================================
The clone has its OWN transaction log — writes to it never touch the source's log at all.

In [0]:
%sql
UPDATE workspace.delta_demo.employees_ep16_clone SET sal = 99999 WHERE eno = 1;

num_affected_rows
1


### Verify — Source Is Completely Untouched

In [0]:
%sql
SELECT * FROM workspace.delta_demo.employees_ep16_source WHERE eno = 1;

eno,ename,sal
1,Ravi,25000.00


###  From this point onward, everything happens only on the SOURCE table. The clone is left untouched

In [0]:
%python
source_eno1 = spark.table("workspace.delta_demo.employees_ep16_source").filter("eno = 1").collect()[0]
clone_eno1 = spark.table("workspace.delta_demo.employees_ep16_clone").filter("eno = 1").collect()[0]

print(f"Source eno=1 salary: {source_eno1['sal']}")
print(f"Clone eno=1 salary: {clone_eno1['sal']}")

if source_eno1['sal'] == 25000 and clone_eno1['sal'] == 99999:
    print("\n✅ VERIFIED: modifying the clone left the source completely untouched.")
else:
    print("\n❌ NOT VERIFIED — investigate.")

Source eno=1 salary: 25000.00
Clone eno=1 salary: 99999.00

✅ VERIFIED: modifying the clone left the source completely untouched.


# =====================================================
# STEP 5 — Force a REAL Physical Deletion on the SOURCE
# =====================================================
🤔 **Prediction:** the clone's log still points at the SOURCE's original physical file — we never copied it. Most Delta tutorials will tell you this means the clone WILL break the moment the source's file gets cleaned up. Let's test that directly, with a real, confirmed deletion — not just a `DRY RUN`.

In [0]:
%sql
UPDATE workspace.delta_demo.employees_ep16_source SET sal = 88888 WHERE eno = 2;

num_affected_rows
1


In [0]:
%sql
OPTIMIZE workspace.delta_demo.employees_ep16_source;

path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 1, true, 0, 0, 1785676485762, 1785676486218, 8, 0, null, List(0, 0), null, 3, 3, 0, 0, null, null)"


In [0]:
%sql
ALTER TABLE workspace.delta_demo.employees_ep16_source SET TBLPROPERTIES ('delta.deletedFileRetentionDuration' = 'interval 0 hours');

In [0]:
%sql
VACUUM workspace.delta_demo.employees_ep16_source;

path
""


### "Does the Clone Still Work After Source VACUUM?"

In [0]:
%python
history_df = spark.sql("DESCRIBE HISTORY workspace.delta_demo.employees_ep16_source")
vacuum_end = history_df.filter(history_df.operation == "VACUUM END") \
    .orderBy("version", ascending=False).first()
metrics = dict(vacuum_end['operationMetrics'])
num_deleted = int(metrics.get('numDeletedFiles', 0))

print(f"numDeletedFiles: {num_deleted}")

if num_deleted > 0:
    print(f"\n✅ VERIFIED: VACUUM genuinely deleted {num_deleted} real")
    print("   physical file(s) from the source. This is not a DRY RUN —")
    print("   actual data was permanently removed from disk.")
else:
    print("\n⚠️ 0 files deleted — nothing was eligible yet. This can")
    print("   happen depending on auto-OPTIMIZE timing. Re-run this cell")
    print("   block, or run another UPDATE + OPTIMIZE + VACUUM cycle,")
    print("   until numDeletedFiles is genuinely greater than 0 before")
    print("   continuing to Step 6.")

numDeletedFiles: 2

✅ VERIFIED: VACUUM genuinely deleted 2 real
   physical file(s) from the source. This is not a DRY RUN —
   actual data was permanently removed from disk.


# =====================================================
# STEP 6 — Now Query the Clone — Does It Still Work?
# =====================================================

In [0]:
%python
try:
    result = spark.table("workspace.delta_demo.employees_ep16_clone").orderBy("eno").collect()
    print("✅ Clone query SUCCEEDED — even after a REAL, confirmed")
    print("   physical deletion on the source:")
    for row in result:
        print(row)
except Exception as e:
    print("❌ Clone query FAILED:")
    print(str(e)[:500])

✅ Clone query SUCCEEDED — even after a REAL, confirmed
   physical deletion on the source:
Row(eno=1, ename='Ravi', sal=Decimal('99999.00'))
Row(eno=2, ename='Sridevi', sal=Decimal('23000.00'))
Row(eno=3, ename='Uma', sal=Decimal('35000.00'))
Row(eno=4, ename='Srik', sal=Decimal('32000.00'))
Row(eno=5, ename='Kanth', sal=Decimal('28000.00'))


## "VACUUM didn't break the clone. What about dropping the source table entirely?"

In [0]:
%sql
DROP TABLE workspace.delta_demo.employees_ep16_source;

## "The source table no longer exists. Let's prove the clone is still an independent Delta table by updating it."

In [0]:
%sql
SELECT *
FROM workspace.delta_demo.employees_ep16_clone;

eno,ename,sal
2,Sridevi,23000.00
3,Uma,35000.00
4,Srik,32000.00
5,Kanth,28000.00
1,Ravi,99999.00


In [0]:
%sql
UPDATE workspace.delta_demo.employees_ep16_clone
SET sal = sal + 100
WHERE eno = 1;

num_affected_rows
1


In [0]:
%sql
SELECT COUNT(*) FROM workspace.delta_demo.employees_ep16_clone;

COUNT(*)
5


In [0]:
%sql
SELECT * FROM workspace.delta_demo.employees_ep16_clone;

eno,ename,sal
1,Ravi,100099.00
2,Sridevi,23000.00
3,Uma,35000.00
4,Srik,32000.00
5,Kanth,28000.00


In [0]:
%sql
UPDATE workspace.delta_demo.employees_ep16_clone
SET sal = sal + 100
WHERE eno = 1;

num_affected_rows
1


### END